In [4]:
from pypdf import PdfReader

# Absolute path to the PDF book on drive D
pdf_path = r"D:\Documents\books\hp\Book 1 - Harry Potter and the Sorcerers Stone.pdf"

# Initialize PDF reader
reader = PdfReader(pdf_path)
print(f"Total pages in PDF: {len(reader.pages)}")

chapter1_text = ""
for page_num in range(0, 206):
    chapter1_text += reader.pages[page_num].extract_text() + "\n"

print(f"Extracted characters: {len(chapter1_text)}")
print("\n--- Preview (First 300 characters) ---")
print(chapter1_text[:300])

Total pages in PDF: 206
Extracted characters: 465818

--- Preview (First 300 characters) ---
 Harry Potter and the Sorcerer’s Stone 
 
By J.K. Rowling 
 
 
 
CHAPTER ONE 
 
 
The Boy Who Lived 
 
      Mr. and Mrs. Dursley, of number four, Privet Drive, were proud to say that they were perfectly 
normal, thank you very much. They were the last people you’d expect to be involved in anything 


### chunk extraction

In [5]:
# Chunking configuration
chunk_size = 500
chunk_overlap = 100

chunks = []
start_idx = 0
text_length = len(chapter1_text)

# Sliding window chunking
while start_idx < text_length:
    end_idx = start_idx + chunk_size
    chunk = chapter1_text[start_idx:end_idx].strip()
    if chunk:
        chunks.append(chunk)
    # Move pointer forward accounting for overlap
    start_idx += chunk_size - chunk_overlap

print(f"Total chunks created: {len(chunks)}")
print("\n--- First Chunk Sample ---")
print(chunks[0])
print("\n--- Second Chunk Sample (Notice the overlap) ---")
print(chunks[1])

Total chunks created: 1165

--- First Chunk Sample ---
Harry Potter and the Sorcerer’s Stone 
 
By J.K. Rowling 
 
 
 
CHAPTER ONE 
 
 
The Boy Who Lived 
 
      Mr. and Mrs. Dursley, of number four, Privet Drive, were proud to say that they were perfectly 
normal, thank you very much. They were the last people you’d expect to be involved in anything 
strange or mysterious, because they just didn’t hold with such nonsense. 
 
      Mr. Dursley was the director of a firm called Grunnings, which made drills. He was a big, beefy 
man with hardly any

--- Second Chunk Sample (Notice the overlap) ---
he director of a firm called Grunnings, which made drills. He was a big, beefy 
man with hardly any neck, although he did have a very large mustache. Mrs. Dursley was thin 
and blonde and had nearly twice the usual amount of neck, which came in very useful as she 
spent so much of her time craning over garden fences, spying on the neighbors. The Dursleys 
had a small son called Dudley and in t

### making embeddings for chuncks and indexing in ChromaDB

In [6]:
import chromadb
from sentence_transformers import SentenceTransformer

# 1. Load lightweight sentence transformer model (~80 MB)
print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Encode all chunks into 384-dimensional dense vectors
print(f"Encoding {len(chunks)} chunks...")
chunk_embeddings = model.encode(chunks, show_progress_bar=True).tolist()

# 3. Initialize PERSISTENT ChromaDB client (Saves on disk)
chroma_client = chromadb.PersistentClient(path=r"D:\projects\harry_potter_rag\chroma_db")

# Create or reset collection
collection_name = "harry_potter_ch1"
existing_collections = [c.name for c in chroma_client.list_collections()]
if collection_name in existing_collections:
    chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)

# 4. Insert documents, vectors, and unique IDs into the vector database
collection.add(
    documents=chunks,
    embeddings=chunk_embeddings,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"Successfully indexed and SAVED {collection.count()} chunks to disk at ./chroma_db.")

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2534.74it/s]


Encoding 1165 chunks...


Batches: 100%|██████████| 37/37 [00:21<00:00,  1.74it/s]


Successfully indexed and SAVED 1165 chunks to disk at ./chroma_db.
